# Silver para Gold — Fato Caged Semestral

Constrói a tabela gerenciada `workspace.gold.fato_caged_semestral` a partir de `workspace.silver.novo_caged_movimentacoes`, na granularidade **ano × semestre × UF × CBO 2002**.

Regras aplicadas:

1. Movimentações agrupadas por semestre, UF e CBO 2002
2. Totais de admissões, desligamentos, movimentações e saldo
3. Somente semestres completos (6 meses presentes, validação global)
4. Duplicidades da Silver e códigos sem descrição são preservados
5. Nenhum dado salarial é utilizado
6. Tabela Delta gerenciada com gravação idempotente (recomputação + substituição atômica)
7. Unicidade da chave validada e totais reconciliados com a Silver antes e depois da gravação

In [0]:
import pyspark.sql.functions as F

In [0]:
# Tabelas de origem e destino
tabela_silver = "workspace.silver.novo_caged_movimentacoes"
tabela_gold = "workspace.gold.fato_caged_semestral"

# Campos usados da Silver (nenhum dado salarial, conforme regra 5)
colunas_origem = [
    "competencia_mov",
    "uf",
    "uf_descricao",
    "cbo2002_ocupacao",
    "cbo2002_ocupacao_descricao",
    "saldo_movimentacao",
]

# Quantidade de meses de um semestre completo
MESES_SEMESTRE = 6

In [0]:
# Lê apenas as colunas necessárias da Silver (sem dados salariais)
# e deriva as dimensões temporais ano/mês/semestre a partir de
# competencia_mov. A Silver NÃO é alterada em nenhuma etapa.
df_movimentacoes = (
    spark.table(tabela_silver)
    .select(*colunas_origem)
    .withColumn("ano", F.year("competencia_mov"))
    .withColumn("mes", F.month("competencia_mov"))
    .withColumn(
        "semestre",
        F.when(F.col("mes") <= 6, F.lit(1)).otherwise(F.lit(2)),
    )
    # Chave de identificação do semestre (usada no filtro de completude)
    .withColumn(
        "ano_semestre",
        F.concat_ws("-", F.col("ano").cast("string"), F.col("semestre").cast("string")),
    )
)

# Amostra para conferência (duplicidades e códigos sem descrição preservados)
display(df_movimentacoes.drop("ano_semestre").limit(5))

In [0]:
# Regra 3: considerar somente semestres completos.
# A presença dos meses é validada GLOBALMENTE (meses distintos de
# competencia_mov no semestre), independente de UF ou CBO.
df_meses_por_semestre = (
    df_movimentacoes
    .groupBy("ano", "semestre")
    .agg(F.countDistinct("mes").alias("meses_presentes"))
    .orderBy("ano", "semestre")
)

display(df_meses_por_semestre)

# Semestres com os 6 meses presentes (mantidos)
semestres_completos = [
    f"{linha['ano']}-{linha['semestre']}"
    for linha in df_meses_por_semestre
    .filter(F.col("meses_presentes") == MESES_SEMESTRE)
    .collect()
]

# Semestres incompletos são descartados, com aviso explícito
for linha in df_meses_por_semestre.filter(
    F.col("meses_presentes") < MESES_SEMESTRE
).collect():
    print(
        f"DESCARTADO: semestre {linha['ano']}/{linha['semestre']} "
        f"com apenas {linha['meses_presentes']} meses presentes"
    )

assert len(semestres_completos) > 0, "Nenhum semestre completo encontrado."

print(
    f"Semestres completos mantidos ({len(semestres_completos)}): "
    + ", ".join(semestres_completos)
)

# Filtra as movimentações somente para os semestres completos
df_filtrado = df_movimentacoes.filter(
    F.col("ano_semestre").isin(semestres_completos)
)

In [0]:
# Regras 1, 2 e 4: agrega por ano, semestre, UF e CBO 2002.
# - admissoes: movimentações com saldo_movimentacao = 1
# - desligamentos: movimentações com saldo_movimentacao = -1
# - total_movimentacoes: todas as linhas contadas (duplicidades preservadas)
# - saldo: soma dos saldos (admissões - desligamentos)
# As descrições são agregadas com max() apenas para carregar o rótulo;
# códigos sem descrição permanecem com descrição nula (regra 4).
df_fato = (
    df_filtrado
    .groupBy("ano", "semestre", "uf", "cbo2002_ocupacao")
    .agg(
        F.max("uf_descricao").alias("uf_descricao"),
        F.max("cbo2002_ocupacao_descricao").alias("cbo2002_ocupacao_descricao"),
        F.sum(F.when(F.col("saldo_movimentacao") == 1, F.lit(1)).otherwise(F.lit(0))).alias("admissoes"),
        F.sum(F.when(F.col("saldo_movimentacao") == -1, F.lit(1)).otherwise(F.lit(0))).alias("desligamentos"),
        F.count(F.lit(1)).alias("total_movimentacoes"),
        F.sum("saldo_movimentacao").alias("saldo"),
    )
)

print("Fato semestral agregada (antes da gravação).")

In [0]:
# Regra 7 (parte 1): apresentação dos resultados ANTES da gravação
total_linhas_fato = df_fato.count()

print(f"Linhas na fato (grão ano × semestre × UF × CBO): {total_linhas_fato:,}")

# Amostra ordenada para conferência
display(
    df_fato
    .orderBy("ano", "semestre", "uf", "cbo2002_ocupacao")
    .limit(20)
)

# Totais gerais da fato para conferência visual
display(
    df_fato.agg(
        F.countDistinct("ano", "semestre").alias("semestres"),
        F.countDistinct("uf").alias("ufs"),
        F.countDistinct("cbo2002_ocupacao").alias("cbos"),
        F.sum("admissoes").alias("admissoes"),
        F.sum("desligamentos").alias("desligamentos"),
        F.sum("total_movimentacoes").alias("movimentacoes"),
        F.sum("saldo").alias("saldo"),
    )
)

In [0]:
# Regra 7 (parte 2): unicidade da chave e reconciliação com a Silver

# a) Unicidade da chave (ano, semestre, uf, cbo2002_ocupacao)
chaves_duplicadas = (
    df_fato
    .groupBy("ano", "semestre", "uf", "cbo2002_ocupacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert chaves_duplicadas == 0, f"{chaves_duplicadas} chaves duplicadas na fato."

# b) saldo_movimentacao só pode ser 1 ou -1 na origem filtrada
registros_saldo_invalido = (
    df_filtrado
    .filter(~F.col("saldo_movimentacao").isin(1, -1))
    .count()
)
assert registros_saldo_invalido == 0, (
    f"{registros_saldo_invalido} registros com saldo diferente de 1/-1."
)

# c) Composição: admissões + desligamentos == total de movimentações
divergencia_composicao = (
    df_fato.filter(
        (F.col("admissoes") + F.col("desligamentos"))
        != F.col("total_movimentacoes")
    )
    .count()
)
assert divergencia_composicao == 0, (
    "admissões + desligamentos difere do total de movimentações."
)

# d) Reconciliação dos totais da fato com a Silver (semestres completos)
resumo_fato = df_fato.agg(
    F.sum("admissoes").alias("admissoes"),
    F.sum("desligamentos").alias("desligamentos"),
    F.sum("total_movimentacoes").alias("movimentacoes"),
    F.sum("saldo").alias("saldo"),
).collect()[0]

resumo_silver = df_filtrado.agg(
    F.sum(
        F.when(F.col("saldo_movimentacao") == 1, F.lit(1)).otherwise(F.lit(0))
    ).alias("admissoes"),
    F.sum(
        F.when(F.col("saldo_movimentacao") == -1, F.lit(1)).otherwise(F.lit(0))
    ).alias("desligamentos"),
    F.count(F.lit(1)).alias("movimentacoes"),
    F.sum("saldo_movimentacao").alias("saldo"),
).collect()[0]

assert resumo_fato["admissoes"] == resumo_silver["admissoes"], "Admissões divergem da Silver."
assert resumo_fato["desligamentos"] == resumo_silver["desligamentos"], "Desligamentos divergem da Silver."
assert resumo_fato["movimentacoes"] == resumo_silver["movimentacoes"], "Movimentações divergem da Silver."
assert resumo_fato["saldo"] == resumo_silver["saldo"], "Saldo diverge da Silver."

print("Validações concluídas com sucesso!")
print(f"  Admissões:     {resumo_fato['admissoes']:,}")
print(f"  Desligamentos: {resumo_fato['desligamentos']:,}")
print(f"  Movimentações: {resumo_fato['movimentacoes']:,}")
print(f"  Saldo:         {resumo_fato['saldo']:,}")

In [0]:
# Regra 6: gravação idempotente na Gold.
# A fato é sempre recomputada a partir da Silver e substituída
# atomicamente (createOrReplace): reexecuções produzem o mesmo
# resultado final, sem duplicar dados.
(
    df_fato.writeTo(tabela_gold)
    .using("delta")
    .createOrReplace()
)

# Documenta a tabela no catálogo
spark.sql(
    f"""COMMENT ON TABLE {tabela_gold} IS
    'Fato semestral do Novo Caged — grão: ano × semestre × UF × CBO 2002.
    Fonte: workspace.silver.novo_caged_movimentacoes. Somente semestres completos
    (6 meses). Sem dados salariais. Construída pelo notebook 030_silver_to_gold_caged.'"""
)

print(f"Tabela {tabela_gold} criada/substituída com sucesso!")

In [0]:
# Regra 7 (parte 3): valida o que efetivamente ficou gravado na Gold
df_gold = spark.table(tabela_gold)

total_gold = df_gold.count()
assert total_gold == total_linhas_fato, (
    "A quantidade gravada na Gold difere da quantidade validada."
)

# Reconferência lendo da tabela gravada (não do DataFrame em memória)
resumo_gold = df_gold.agg(
    F.sum("admissoes").alias("admissoes"),
    F.sum("desligamentos").alias("desligamentos"),
    F.sum("total_movimentacoes").alias("movimentacoes"),
    F.sum("saldo").alias("saldo"),
).collect()[0]

assert resumo_gold["admissoes"] == resumo_fato["admissoes"], "Admissões divergem após gravação."
assert resumo_gold["desligamentos"] == resumo_fato["desligamentos"], "Desligamentos divergem após gravação."
assert resumo_gold["movimentacoes"] == resumo_fato["movimentacoes"], "Movimentações divergem após gravação."
assert resumo_gold["saldo"] == resumo_fato["saldo"], "Saldo diverge após gravação."

print(f"Registros gravados na Gold: {total_gold:,}")
print("Reconciliação Gold × Silver concluída com sucesso!")

# Amostra da tabela gravada
display(
    df_gold
    .orderBy("ano", "semestre", "uf", "cbo2002_ocupacao")
    .limit(20)
)

# Histórico de transações Delta (idempotência: uma versão por execução)
display(
    spark.sql(f"DESCRIBE HISTORY {tabela_gold}")
    .select("version", "timestamp", "operation", "operationParameters")
    .limit(10)
)